# LightGBMRanker: A Reusable Ranker for Recommendation

This notebook demonstrates how to build a ranking pipeline for recommendations using `LightGBMRanker`, a utility class that wraps LightGBM training, inference, save, and load into a single sklearn-like interface.

```python
from recommenders.models.lightgbm import LightGBMRanker

ranker = LightGBMRanker(params={...})
ranker.fit(train_x, train_y, valid_x, valid_y)
scores  = ranker.predict(test_x)
ranker.save("model.lgb")

loaded  = LightGBMRanker.load("model.lgb")
scores  = loaded.predict(test_x)
```

## 0. Imports and Parameters

In [ ]:
import os
import sys
import numpy as np
from tempfile import TemporaryDirectory
from sklearn.metrics import roc_auc_score, log_loss

import recommenders.datasets.criteo as criteo
from recommenders.models.lightgbm import LightGBMRanker
from recommenders.models.lightgbm.lightgbm_utils import NumEncoder

print(f"Python: {sys.version}")

In [ ]:
# Data settings
SIZE      = "sample"  # 'sample' | 'full'
LABEL_COL = "Label"
NUME_COLS = [f"I{i}" for i in range(1, 14)]
CATE_COLS = [f"C{i}" for i in range(1, 27)]
HEADER    = [LABEL_COL] + NUME_COLS + CATE_COLS

# LightGBMRanker parameters (merged on top of DEFAULT_PARAMS)
PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "num_leaves": 64,
    "learning_rate": 0.15,
    "feature_fraction": 0.8,
    "num_threads": 4,
}
NUM_BOOST_ROUND       = 100
EARLY_STOPPING_ROUNDS = 20

## 1. Data Preparation

We use the [Criteo dataset](https://www.kaggle.com/c/criteo-display-ad-challenge) — a standard industry benchmark for CTR prediction — and split it chronologically into train (80%), validation (10%), and test (10%) sets.

In [ ]:
with TemporaryDirectory() as tmp:
    all_data = criteo.load_pandas_df(size=SIZE, local_cache_path=tmp, header=HEADER)

length     = len(all_data)
train_data = all_data.iloc[: int(0.8 * length)]
valid_data = all_data.iloc[int(0.8 * length) : int(0.9 * length)]
test_data  = all_data.iloc[int(0.9 * length) :]

print(f"train: {len(train_data):,}  valid: {len(valid_data):,}  test: {len(test_data):,}")

## 2. Feature Encoding (NumEncoder)

`NumEncoder` converts all categorical columns into dense numerical features through three steps:
1. **Low-frequency filtering** — rare categories become `<LESS>`, missing values become `<UNK>`.
2. **Sequential target & count encoding** — encodes each sample using statistics from previous samples only (no label leakage).
3. **Binary encoding** — the ordinal-encoded category values are expanded into bit vectors.

In [ ]:
encoder = NumEncoder(CATE_COLS, NUME_COLS, LABEL_COL)

train_x, train_y = encoder.fit_transform(train_data)
valid_x, valid_y = encoder.transform(valid_data)
test_x,  test_y  = encoder.transform(test_data)

print(f"train_x: {train_x.shape}  valid_x: {valid_x.shape}  test_x: {test_x.shape}")

## 3. Training (LightGBMRanker.fit)

Passing a validation set enables early stopping automatically.

In [ ]:
ranker = LightGBMRanker(
    params=PARAMS,
    num_boost_round=NUM_BOOST_ROUND,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)

ranker.fit(train_x, train_y, valid_x=valid_x, valid_y=valid_y)

## 4. Inference and Evaluation (LightGBMRanker.predict)

In [ ]:
scores = ranker.predict(test_x)

auc     = roc_auc_score(test_y.reshape(-1), scores)
logloss = log_loss(test_y.reshape(-1), scores)

print(f"AUC:     {auc:.4f}")
print(f"LogLoss: {logloss:.4f}")

## 5. Top-K Ranking Example

In a real recommendation pipeline, you score all candidate items per user and sort by descending score.

In [ ]:
import pandas as pd

result_df = pd.DataFrame({
    "score": scores,
    "label": test_y.reshape(-1),
})

top5 = result_df.nlargest(5, "score")
print("Top-5 recommendations (sorted by score):")
display(top5)

## 6. Save and Load (save / load)

A model saved with `save()` can be restored with `LightGBMRanker.load()` and produces identical predictions.

In [ ]:
with TemporaryDirectory() as tmp:
    model_path = os.path.join(tmp, "ranker.lgb")

    ranker.save(model_path)

    loaded_ranker = LightGBMRanker.load(model_path)
    loaded_scores = loaded_ranker.predict(test_x)

assert np.allclose(scores, loaded_scores), "Predictions differ after save/load!"
print("Save/load verified: predictions are identical.")

## 7. Learning-to-Rank (lambdarank) Mode

For query-level ranking (e.g. ranking items per user session), switch the objective to `lambdarank` and supply a group array that specifies how many items belong to each query.

```python
import lightgbm as lgb
from recommenders.models.lightgbm import LightGBMRanker

ltr_params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [5, 10],
    "num_leaves": 64,
    "learning_rate": 0.05,
}

# train_group: list of item counts per user, e.g. [3, 5, 4]
lgb_train = lgb.Dataset(train_x, train_y.reshape(-1))
lgb_train.set_group(train_group)

ltr_ranker = LightGBMRanker(params=ltr_params, num_boost_round=200)
ltr_ranker.model = lgb.train(ltr_params, lgb_train, num_boost_round=200)

ltr_scores = ltr_ranker.predict(test_x)
```